# LLM O*NET Mapping — Top 10 Companies (Opus, no --output-format json)

Same as original `map_top10_companies_to_onet_LLM_based.ipynb` with explicit `--model opus`.

Purpose: reproduce original Step 2 run (Opus, 41min) in current environment for fair comparison.

Comparison:
- `map_all_companies_to_onet_with_cost.ipynb`: Sonnet + `--output-format json`
- `map_top10_sonnet_no_json.ipynb`: Sonnet, no json
- **This notebook**: Opus, no json

In [1]:
import pandas as pd
import subprocess
import json
import re
import time
import os
from collections import defaultdict
from pathlib import Path

In [ ]:
NOTEBOOK_DIR = Path('.').resolve()
DATA_ROOT = NOTEBOOK_DIR / '../../../../data/processed/llm_title_accuracy_test'
TITLES_PATH = DATA_ROOT / 'step-1-extract-unique-titles/company_unique_titles.csv'
TAXONOMY_PATH = NOTEBOOK_DIR / '../../../../data/raw/onet_job_occupation_taxonomy.csv'
OUTPUT_DIR = DATA_ROOT / 'step-2-llm-onet-mapping/benchmark_opus_no_json/output'
TEMP_PROMPT = NOTEBOOK_DIR / 'temp_prompt.txt'

OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

BATCH_SIZE = 30
TOP_N = 10

print(f'Notebook dir : {NOTEBOOK_DIR}')
print(f'Data root    : {DATA_ROOT}')
print(f'Titles       : {TITLES_PATH}')
print(f'Taxonomy     : {TAXONOMY_PATH}')
print(f'Output       : {OUTPUT_DIR}')
print(f'Temp prompt  : {TEMP_PROMPT}')

In [3]:
df_titles = pd.read_csv(TITLES_PATH)
df_taxonomy = pd.read_csv(TAXONOMY_PATH)

onet_titles = df_taxonomy['Title'].tolist()
top_companies = df_titles.head(TOP_N)

print(f'Total companies  : {len(df_titles):,}')
print(f'O*NET categories : {len(onet_titles):,}')
print(f'\nCompanies to process ({TOP_N}):')
print(top_companies[['company_name', 'total_postings', 'unique_titles_count']].to_string(index=False))

Total companies  : 24,428
O*NET categories : 923

Companies to process (10):
                                  company_name  total_postings  unique_titles_count
                               The Job Network            1003                  724
                                    TEKsystems             529                  395
                                          Dice             415                  385
                                Insight Global             418                  359
                                        Macy's             333                  306
                                VolunteerMatch             322                  292
                                  Apex Systems             325                  287
                                        Amazon             343                  286
Liberty Healthcare and Rehabilitation Services            1108                  265
                     Maxim Healthcare Staffing             278                  247

In [4]:
def map_titles_for_company(company_row, onet_titles, batch_size=BATCH_SIZE):
    company_name = company_row['company_name']
    titles_list = company_row['unique_titles'].split(';')

    print(f'\nProcessing: {company_name}')
    print(f'  Unique titles: {len(titles_list)}')

    all_mappings = []
    total = len(titles_list)

    for i in range(0, total, batch_size):
        batch = titles_list[i:i+batch_size]
        batch_num = i // batch_size + 1
        total_batches = (total - 1) // batch_size + 1
        print(f'  Batch {batch_num}/{total_batches}: {i+1}-{min(i+batch_size, total)} / {total}')

        prompt = f"""Map each job title to the most relevant O*NET category.

Job titles to map:
{json.dumps(batch)}

Available O*NET categories:
{json.dumps(onet_titles)}

Return JSON only, no explanation:
{{"mappings": [{{"raw": "original title", "onet": "O*NET category"}}]}}
"""

        with open(TEMP_PROMPT, 'w', encoding='utf-8') as f:
            f.write(prompt)

        result = subprocess.run(
            f'type "{TEMP_PROMPT}" | claude --print --model opus -',
            capture_output=True, text=True, shell=True
        )

        json_match = re.search(r'\{.*\}', result.stdout, re.DOTALL)
        if json_match:
            mappings = json.loads(json_match.group())
            all_mappings.extend(mappings['mappings'])
        else:
            print(f'    Warning: Failed to parse batch')

        time.sleep(1)

    print(f'  ✓ Mapped {len(all_mappings)} titles')

    grouped = defaultdict(list)
    for m in all_mappings:
        grouped[m['onet']].append(m['raw'])

    output_data = []
    for onet_tag, titles in grouped.items():
        output_data.append({
            'onet_tag': onet_tag,
            'company_name': company_name,
            'raw_titles': ';'.join(titles)
        })

    return pd.DataFrame(output_data)

In [5]:
import datetime

all_results = []

start_time = datetime.datetime.now()
print(f'Start: {start_time.strftime("%H:%M:%S")}')

for idx, row in top_companies.iterrows():
    company_name = row['company_name']

    output_file = OUTPUT_DIR / f"{company_name.replace('/', '_')}_tagged.csv"
    if output_file.exists():
        print(f'\nSkipping {company_name} (already processed)')
        df_existing = pd.read_csv(output_file)
        all_results.append(df_existing)
        continue

    df_result = map_titles_for_company(row, onet_titles)

    df_result.to_csv(output_file, index=False, encoding='utf-8-sig')
    print(f'  ✓ Saved to {output_file}')

    all_results.append(df_result)

    if idx < len(top_companies) - 1:
        time.sleep(2)

elapsed = datetime.datetime.now() - start_time
print(f'\nDone: {datetime.datetime.now().strftime("%H:%M:%S")}')
print(f'Elapsed: {elapsed}')

Start: 17:07:42

Processing: The Job Network
  Unique titles: 726
  Batch 1/25: 1-30 / 726
  Batch 2/25: 31-60 / 726
  Batch 3/25: 61-90 / 726
  Batch 4/25: 91-120 / 726
  Batch 5/25: 121-150 / 726
  Batch 6/25: 151-180 / 726
  Batch 7/25: 181-210 / 726
  Batch 8/25: 211-240 / 726
  Batch 9/25: 241-270 / 726
  Batch 10/25: 271-300 / 726
  Batch 11/25: 301-330 / 726
  Batch 12/25: 331-360 / 726
  Batch 13/25: 361-390 / 726
  Batch 14/25: 391-420 / 726
  Batch 15/25: 421-450 / 726
  Batch 16/25: 451-480 / 726
  Batch 17/25: 481-510 / 726
  Batch 18/25: 511-540 / 726
  Batch 19/25: 541-570 / 726
  Batch 20/25: 571-600 / 726
  Batch 21/25: 601-630 / 726
  Batch 22/25: 631-660 / 726
  Batch 23/25: 661-690 / 726
  Batch 24/25: 691-720 / 726
  Batch 25/25: 721-726 / 726
  ✓ Mapped 726 titles
  ✓ Saved to C:\Users\blitz\Desktop\Self-Binnacle\Voyages\Projects\Class-INFO3220-systems-project\resources\linkedIn_JDs\gilJOBi\data\processed\llm_title_accuracy_test\step-2-llm-onet-mapping\benchmark_op

KeyboardInterrupt: 

In [ ]:
df_combined = pd.concat(all_results, ignore_index=True)

combined_output = OUTPUT_DIR / 'all_companies_tagged.csv'
df_combined.to_csv(combined_output, index=False, encoding='utf-8-sig')

print(f'✓ Combined results saved to {combined_output}')
print(f'  Total companies   : {df_combined["company_name"].nunique()}')
print(f'  Total O*NET tags  : {len(df_combined)}')
print(f'  Unique O*NET tags : {df_combined["onet_tag"].nunique()}')

In [ ]:
if TEMP_PROMPT.exists():
    TEMP_PROMPT.unlink()
    print('✓ Cleaned up temp_prompt.txt')